# Sysmon Malware Detection — เทรนบน Colab

ใช้โค้ด `host/ml_train.py` + `host/ml_predict.py` จากรีโป

**ต้องเตรียมเอง 2 อย่าง** (เซลล์ที่ 2 กับ 3):
1. ไฟล์โค้ด `ml_train.py` `ml_predict.py`
2. `merged_dataset.csv` (~50 MB) — ไม่ได้อยู่ใน git เพราะ `host/dataset/` ถูก gitignore

ไม่ต้อง `pip install` อะไรเลย — Colab มี pandas / scikit-learn / joblib ครบแล้ว

In [ ]:
# 1) เช็คเวอร์ชัน — โค้ดทดสอบแล้วบน pandas 2.x และ 3.x, sklearn 1.6-1.9
import sys, pandas, sklearn, numpy, joblib
print(sys.version)
for m in (pandas, sklearn, numpy, joblib):
    print(f"{m.__name__:12s} {m.__version__}")

## 2) เอาโค้ดขึ้น Colab

**`git clone` ตรงๆ ไม่ได้** เพราะรีโป `Anochaakimbo/sysmon_lab_environment`
เป็น private (ยิง GitHub API แบบไม่ล็อกอินได้ 404)
และ branch `windows-scenarios-art` ที่มีไฟล์ ML **ยังไม่ได้ push ขึ้น GitHub เลย**

เลือกวิธีใดวิธีหนึ่ง:

| วิธี | เหมาะเมื่อ |
|------|-----------|
| **A. อัปโหลดไฟล์** (default) | ง่ายสุด ไม่ต้องยุ่งกับ token — ไฟล์แค่ 2 ตัว ~30 KB |
| **B. Google Drive** | ไม่อยากอัปโหลดซ้ำทุกครั้งที่ต่อ runtime ใหม่ |
| **C. git clone** | ต้อง `git push` branch ขึ้นก่อน + ใช้ GitHub token |

In [ ]:
# --- วิธี A: อัปโหลด ml_train.py + ml_predict.py (เลือกทีเดียวสองไฟล์ได้) ---
import os, shutil
os.makedirs("/content/host/dataset", exist_ok=True)
os.chdir("/content")

from google.colab import files
up = files.upload()
for name in up:
    shutil.move(name, f"/content/host/{name}")

!ls -l /content/host/

In [ ]:
# --- วิธี B: จาก Google Drive (ก๊อปทั้งโฟลเดอร์ host ขึ้น Drive ไว้ก่อน) ---
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/host/dataset
# !cp /content/drive/MyDrive/sysmon-lab/host/ml_*.py /content/host/
# %cd /content

# --- วิธี C: git clone (ต้อง push branch ขึ้น GitHub ก่อน) ---
#   บนเครื่องตัวเอง:  git push -u origin windows-scenarios-art
#   แล้วสร้าง Personal Access Token (classic, scope repo) ที่
#   https://github.com/settings/tokens
# from getpass import getpass
# TOKEN = getpass('GitHub token: ')
# !git clone -q -b windows-scenarios-art \
#     https://{TOKEN}@github.com/Anochaakimbo/sysmon_lab_environment.git /content/repo
# %cd /content/repo
#   หมายเหตุ: อย่าพิมพ์ token ลงในเซลล์ตรงๆ notebook จะเก็บติดไปด้วย ใช้ getpass เสมอ

In [ ]:
# 3) อัปโหลด merged_dataset.csv (~50 MB) ลง host/dataset/
import os, shutil
os.makedirs("/content/host/dataset", exist_ok=True)
os.chdir("/content")

from google.colab import files
up = files.upload()
for name in up:
    shutil.move(name, "/content/host/dataset/merged_dataset.csv")

# ถ้าไฟล์อยู่บน Drive แล้วให้ใช้บรรทัดนี้แทน (เร็วกว่ามาก)
# !cp "/content/drive/MyDrive/merged_dataset.csv" /content/host/dataset/merged_dataset.csv

!ls -lh /content/host/dataset/merged_dataset.csv

## เทรน

3 โปรโตคอลที่ควรรายงานในธีสิส (ตัวเลขต่างกันได้ 40 จุดบนข้อมูลชุดเดียวกัน
ถ้าไม่ระบุว่าใช้แบบไหน ตัวเลขไม่มีความหมาย):

| split | คืออะไร |
|-------|---------|
| `process` | GroupSplit ตาม ProcessGuid — **ตัวเลขหลัก** |
| `scenario` | leave-scenario-out — ไม่เคยเห็นการโจมตีชนิดนั้น (zero-day) |
| `paper` | ตาม Fig.4 ของเปเปอร์ — unsupervised เทรน benign ล้วน |

ผลต่อท้ายลง `reference/ml_results.csv` ทุกครั้ง โมเดลลง `models/*.pkl`

In [ ]:
# 4) ตัวเลขหลัก — process-level, group by ProcessGuid, + โมเดลบอกตระกูลมัลแวร์
!python host/ml_train.py --level process --split process --family

In [ ]:
# 5) เทียบกับเปเปอร์ตรงๆ — unsupervised เทรนด้วย benign อย่างเดียว
#    ดู AUC เป็นหลัก ไม่ใช่ F1 (test set เป็น malware 83% -> baseline flag ทุกแถวได้ F1 0.908)
!python host/ml_train.py --level event --split paper

In [ ]:
# 6) zero-day — เทรนด้วยการโจมตีชุดหนึ่ง แล้วเทสกับชนิดที่ไม่เคยเห็น
!python host/ml_train.py --level process --split scenario

In [ ]:
# 7) รวมผลทั้งหมดเป็นตารางเดียว (เอาไปวางในธีสิสได้)
import pandas as pd
r = pd.read_csv("reference/ml_results.csv")
cols = ["Learning_Type", "Protocol", "Level", "Algorithm",
        "Accuracy", "Precision", "Recall", "F1", "AUC"]
display(r[cols].round(4))

# เทียบกับเปเปอร์ (Achmad et al. 2025)
paper = pd.DataFrame([
    ("Supervised", "Naive Bayes", 0.3810), ("Supervised", "Decision Tree", 0.8720),
    ("Supervised", "Random Forest", 0.8868), ("Supervised", "SVM", 0.5906),
    ("Unsupervised", "Isolation Forest", 0.7620),
    ("Unsupervised", "Local Outlier Factor", 0.9750),
    ("Unsupervised", "One-Class SVM", 0.8051),
], columns=["Learning_Type", "Algorithm", "F1_paper"])
display(r.merge(paper, on=["Learning_Type", "Algorithm"])
         [["Protocol", "Level", "Algorithm", "F1", "F1_paper", "AUC"]].round(4))

## เอาโมเดลไปตรวจ log ที่ไม่ได้อยู่ในชุด train

In [ ]:
# 8) อัปโหลด CSV ที่จะตรวจ แล้วยิงเข้าโมเดล
import shutil, os
os.chdir("/content")
from google.colab import files
up = files.upload()
target = f"/content/{list(up)[0]}"

!python host/ml_predict.py models/process_process.pkl "{target}" --out pred.csv

import pandas as pd
pred = pd.read_csv("pred.csv")
display(pred.sort_values("score", ascending=False).head(20))

In [ ]:
# 9) ดาวน์โหลดโมเดล + ผลกลับเครื่อง
!zip -qr out.zip models reference/ml_results.csv
from google.colab import files
files.download("out.zip")

## หมายเหตุ

- **runtime ของ Colab ตายแล้วไฟล์หายหมด** ต้องอัปโหลดใหม่ทุกครั้ง
  ถ้าจะรันบ่อยให้ใช้วิธี B (Drive) จะไม่ต้องอัปซ้ำ
- **`--split random` ห้ามใช้รายงาน** — event หลายแถวมาจาก process เดียวกัน
  สุ่มแล้วรั่วข้าม train/test ได้ F1 0.98 ปลอมๆ (เปเปอร์ใช้แบบนี้)
- **ดู `[baseline] flag ทุกแถว` ทุกครั้ง** — โมเดล unsupervised ที่ไม่ชนะบรรทัดนี้
  แปลว่าไม่ได้ตรวจจับอะไรเลย
- **SVM (rbf) ช้า** — ถูกจำกัดแถวเทรนไว้ที่ 20,000 ปรับด้วย `--svm-max-train`
- CPU ของ Colab ช้ากว่าเครื่องตัวเอง เริ่มจาก `--level process` (23,971 แถว)
  ก่อน `--level event` (71,086 แถว) — ไม่ต้องใช้ GPU ทั้งหมดเป็น sklearn ล้วน
- รายละเอียดว่าทำไมโปรโตคอลถึงสำคัญ: `docs/ml_results_review.md`